# Recommender System - Collaborative Filtering
This notebook documents a practical user-based Collaborative Filtering pipeline using the MovieLens 100K dataset.
It complements a recommendation systems presentation and focuses on implementation decisions and outputs.



## Objective
Build a user-item matrix from ratings.
Compute user-to-user similarity.
Generate top-N recommendations for a target user.


In [1]:
from src import (load_ratings, load_movies, build_user_item_matrix,
    fill_missing_values, compute_sparsity, compute_user_similarity,
    recommend_user_based)


## Data Loading
MovieLens 100K provides user IDs, item IDs, and explicit ratings.
Movies metadata supplies titles so recommendations are readable.
The dataset is structured for quick joins between ratings and items.


In [2]:
ratings_df = load_ratings("datasets/ratings.csv")
movies_df = load_movies("datasets/movies.csv")

In [3]:
from IPython.display import display
display(ratings_df.head())
display(movies_df.head())


,user_id,item_id,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


,item_id,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy



## Data Exploration
Exploration validates data integrity and shapes before modeling.
We check the number of users and items, then inspect rating distribution to understand bias in feedback frequency.


In [4]:
ratings_df.head()
ratings_df.shape

(100836, 4)


## Sparsity Analysis
Sparsity is the fraction of missing user-item ratings in the matrix.
Collaborative Filtering relies on user interactions, but in real datasets most entries are missing, which makes similarity harder to compute.
High sparsity reduces overlap and weakens similarity signals.


In [5]:
sparsity = compute_sparsity(ratings_df)


## User-Item Matrix
Rows represent users and columns represent items.
Each cell holds a rating, with missing values treated as "not rated."
This matrix is the core structure for computing similarities and generating recommendations.


In [6]:
user_item_matrix = build_user_item_matrix(ratings_df)

user_item_filled = fill_missing_values(user_item_matrix)


## Similarity Computation
User similarity is computed by comparing rating vectors.
Cosine similarity measures the angle between vectors, focusing on preference alignment rather than rating scale.
Higher similarity indicates more aligned taste.


In [7]:
user_similarity = compute_user_similarity(user_item_filled)


## Recommendation Logic
Select a target user.
Find similar users based on cosine similarity.
Aggregate similar users' ratings with a weighted average to score unseen items.
Rank scores to produce top-N recommendations.


In [8]:
recommendations = recommend_user_based(
    user_id=1,
    user_item_matrix=user_item_filled,
    similarity_matrix=user_similarity,
    movies_df=movies_df
)

recommendations.head()

,item_id,score,title
0,2067,5.0,Doctor Zhivago (1965)
1,4014,5.0,Chocolat (2000)
2,4235,5.0,Amores Perros (Love's a Bitch) (2000)
3,2106,5.0,Swing Kids (1993)
4,2398,5.0,Miracle on 34th Street (1947)


## Example
This section shows a single user's top-rated movies and the top recommendations generated by user-based collaborative filtering.
It's meant as a quick sanity check that the pipeline is producing readable, sensible outputs before tuning hyperparameters.


In [9]:
target_user = 1

# Show movies the user has already rated (top 10 by rating)
user_ratings = ratings_df[ratings_df["user_id"] == target_user]
user_ratings = user_ratings.merge(
    movies_df[["item_id", "title"]],
    on="item_id",
    how="left"
)
user_ratings = user_ratings.sort_values("rating", ascending=False).head(10)

from IPython.display import display
print(f"Top ratings by user {target_user}")
display(user_ratings[["title", "rating"]])

# Show recommendations
print(f"Top recommendations for user {target_user}")
display(recommendations[["title", "score"]].head(10))


Top ratings by user 1


,title,rating
3,Seven (a.k.a. Se7en) (1995),5.0
4,"Usual Suspects, The (1995)",5.0
6,Bottle Rocket (1996),5.0
13,Dumb & Dumber (Dumb and Dumber) (1994),5.0
11,Billy Madison (1995),5.0
10,Desperado (1995),5.0
9,Canadian Bacon (1995),5.0
8,Rob Roy (1995),5.0
35,Pinocchio (1940),5.0
31,Tombstone (1993),5.0


Top recommendations for user 1


,title,score
0,Doctor Zhivago (1965),5.0
1,Chocolat (2000),5.0
2,Amores Perros (Love's a Bitch) (2000),5.0
3,Swing Kids (1993),5.0
4,Miracle on 34th Street (1947),5.0
5,Scott Pilgrim vs. the World (2010),5.0
6,"Legend of Drunken Master, The (Jui kuen II) (1...",5.0
7,"Princess and the Warrior, The (Krieger und die...",5.0
8,Wet Hot American Summer (2001),5.0
9,"Last Picture Show, The (1971)",5.0


## Limitation
User-based CF struggles with cold-start users and items, and it can be sensitive to sparsity.
Similarity scores may be noisy when users share very few co-rated items, which can reduce recommendation quality.
